In [ ]:
import pandas as pd
import os

def create_verified_csv(source_file, target_file, output_csv):
    print(f"🔄 İşleniyor: {source_file} + {target_file} -> {output_csv}")

    if not os.path.exists(source_file) or not os.path.exists(target_file):
        print(f"❌ HATA: Dosyalar bulunamadı! Lütfen .txt dosyalarını yüklediğinden emin ol.")
        return

    # 1. Kaynak (Hatalı) Dosyayı Oku
    with open(source_file, "r", encoding="utf-8") as f:
        # Sadece "S " ile başlayan satırları al ve başındaki "S "yi sil
        sources = [line.strip()[2:] for line in f if line.startswith("S ")]

    # 2. Hedef (Doğru) Dosyayı Oku
    with open(target_file, "r", encoding="utf-8") as f:
        # Boş satırları atla
        targets = [line.strip() for line in f if line.strip()]

    # Satır sayısı kontrolü
    min_len = min(len(sources), len(targets))
    sources = sources[:min_len]
    targets = targets[:min_len]

    # 3. DataFrame Oluştur
    df = pd.DataFrame({"input_text": sources, "target_text": targets})

    # 4. KONTROL AŞAMASI (En Önemli Kısım)
    # Girdi ve Çıktının FARKLI olduğu satırları say
    diff_count = len(df[df["input_text"] != df["target_text"]])

    print(f"   📊 Toplam Satır: {len(df)}")
    print(f"   ✅ Düzeltme İçeren Satır Sayısı: {diff_count}")

    if diff_count == 0:
        print("   ⚠️ UYARI: Bu dosya tamamen kopya! Bir sorun var.")
    else:
        # CSV Olarak Kaydet
        df.to_csv(output_csv, index=False)
        print(f"   💾 {output_csv} başarıyla oluşturuldu.")

# Dosyaları Oluştur
create_verified_csv("boun_source_train.txt", "boun_target_train.txt", "train.csv")
create_verified_csv("boun_source_dev.txt", "boun_target_dev.txt", "val.csv")
create_verified_csv("boun_source_test.txt", "boun_target_test.txt", "test.csv")

🔄 İşleniyor: boun_source_train.txt + boun_target_train.txt -> train.csv
   📊 Toplam Satır: 7516
   ✅ Düzeltme İçeren Satır Sayısı: 3756
   💾 train.csv başarıyla oluşturuldu.
🔄 İşleniyor: boun_source_dev.txt + boun_target_dev.txt -> val.csv
   📊 Toplam Satır: 1842
   ✅ Düzeltme İçeren Satır Sayısı: 921
   💾 val.csv başarıyla oluşturuldu.
🔄 İşleniyor: boun_source_test.txt + boun_target_test.txt -> test.csv
   📊 Toplam Satır: 1017
   ✅ Düzeltme İçeren Satır Sayısı: 507
   💾 test.csv başarıyla oluşturuldu.


In [ ]:
import pandas as pd
import torch
import os
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from torch.utils.data import Dataset
from google.colab import drive

# 1. HAZIRLIK
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

MODEL_NAME = "google/mt5-small"
OUTPUT_DIR = "/content/drive/MyDrive/grammer_model_final_fix2"

# 2. VERİ SETİ SINIFI (En Basit ve Hatasız Hali)
# Burada padding yapmıyoruz! Sadece token ID'lerini döndürüyoruz.
# Padding'i DataCollator yapacak (Dynamic Padding).
class GecDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.data = df
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # 1. Girdiyi Tokenize Et (Padding YOK, Truncation VAR)
        model_inputs = self.tokenizer(
            row["input_text"],
            max_length=64,
            truncation=True
        )

        # 2. Hedefi (Label) Tokenize Et
        # text_target parametresi en güncel yöntemdir.
        labels = self.tokenizer(
            text_target=row["target_text"],
            max_length=64,
            truncation=True
        )

        return {
            "input_ids": model_inputs["input_ids"],
            "attention_mask": model_inputs["attention_mask"],
            "labels": labels["input_ids"]
        }

# 3. VERİ YÜKLEME VE KONTROL
def load_data(file_name):
    try:
        df = pd.read_csv(file_name).dropna().astype(str)
        # Sadece değişmesi gereken cümleleri alalım
        df = df[df["input_text"] != df["target_text"]]
        # Boş cümleleri atalım (Loss 0 hatasının ana kaynağı olabilir)
        df = df[df["input_text"].str.len() > 1]
        df = df[df["target_text"].str.len() > 1]
        return df
    except:
        return pd.DataFrame()

train_df = load_data("train.csv")
val_df = load_data("val.csv")

print(f"🔥 Temizlenmiş Veri Sayısı: {len(train_df)}")
if len(train_df) == 0: raise ValueError("Veri seti boş!")

# 4. MODEL VE TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, legacy=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Dataset Oluşturma
train_ds = GecDataset(train_df, tokenizer)
val_ds = GecDataset(val_df, tokenizer)

# 5. DATA COLLATOR (SİHİRLİ KISIM)
# Bu arkadaş, gelen verileri o anki batch'teki en uzun cümleye göre pad'ler.
# label_pad_token_id=-100 yaparak padding'leri loss hesabından otomatik düşer.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    padding=True # Dinamik padding aktif
)

# 6. EĞİTİM AYARLARI (ADAFACTOR + DOĞRU LR)
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",

    # Optimizer Ayarları
    optim="adafactor",
    learning_rate=5e-4, # Adafactor için ideal

    per_device_train_batch_size=8, # Batch size'ı biraz artırabiliriz
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    predict_with_generate=True,

    fp16=False,         # KAPALI (mT5 için zorunlu)
    max_grad_norm=1.0,  # Clipping AÇIK

    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

# Kontrol: İlk örnek neye benziyor?
print("🔍 İlk örneğin Label uzunluğu:", len(train_ds[0]['labels']))
# Loss 0 olmaması için labels içinde -100 olmayan değerler olmalı.

print("🚀 Eğitim Başlıyor...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Bitti!")

Mounted at /content/drive
🔥 Temizlenmiş Veri Sayısı: 3756


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

🔍 İlk örneğin Label uzunluğu: 31
🚀 Eğitim Başlıyor...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.400400,0.200815
2,0.238100,0.135770
3,0.241600,0.121484


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Bitti!


In [4]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
import os

from google.colab import drive

# 1. Önce Drive'ı Garantiye Alalım (Bağlı değilse bağlar)
if not os.path.exists('/content/drive'):
    print("🔌 Drive bağlanıyor...")
    drive.mount('/content/drive')
else:
    print("✅ Drive zaten bağlı.")
# ---------------------------------------------------------
# 1. AYARLAR (YOLU DÜZELTTİK)
# ---------------------------------------------------------
# ARTIK TAM ADRESİ VERİYORUZ:
MODEL_PATH = "/content/drive/MyDrive/grammer_model_final_fix2"

TEST_FILE = "test.csv"
OUTPUT_FILE = "test_sonuclari.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Çalışma cihazı: {device}")

# ---------------------------------------------------------
# 2. MODELİ YÜKLEME
# ---------------------------------------------------------
print(f"📂 Model yükleniyor: {MODEL_PATH}...")

if not os.path.exists(MODEL_PATH):
    raise ValueError(f"❌ HATA: Klasör bulunamadı! Lütfen Drive yolunu kontrol et: {MODEL_PATH}")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device)
    model.eval()
    print("✅ Model başarıyla yüklendi! (Checkpoint-470)")
except Exception as e:
    print(f"❌ Model yükleme hatası: {e}")
    raise e # Hatayı gizleme, işlemi durdur

# ---------------------------------------------------------
# 3. TEST VERİSİNİ HAZIRLAMA
# ---------------------------------------------------------
if os.path.exists(TEST_FILE):
    df_test = pd.read_csv(TEST_FILE).dropna().astype(str)
    print(f"📄 Test dosyası okundu: {len(df_test)} satır.")
else:
    print("⚠️ test.csv bulunamadı! Örnek verilerle test ediliyor...")
    df_test = pd.DataFrame({
        "input_text": [
            "Ben eve gidiyor.",
            "Yarın okul gittim.",
            "Hava çok güzel bugun.",
            "Bu araba çok hızlı gidiyorlar.",
            "Ben gelmeyeceğim çünkü hastayım."
        ],
        "target_text": [""] * 5 # Hedef metin yoksa boş bırak
    })

# ---------------------------------------------------------
# 4. TAHMİN DÖNGÜSÜ
# ---------------------------------------------------------
print("🚀 Düzeltme işlemi başlıyor...")

predictions = []
BATCH_SIZE = 16 # Hızlanmak için toplu işlem (Batching)

# Veriyi batch'ler halinde işleyelim (Daha hızlı olur)
inputs_list = df_test["input_text"].tolist()

for i in tqdm(range(0, len(inputs_list), BATCH_SIZE), desc="İşleniyor"):
    batch_texts = inputs_list[i : i + BATCH_SIZE]

    # Batch Tokenization
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        max_length=64,
        truncation=True,
        padding=True
    ).to(device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=64,
            num_beams=5,
            early_stopping=True
        )

    # Decode
    batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(batch_preds)

# Sonuçları kaydet
df_test["model_output"] = predictions

# Ekrana bas
print("\n👀 ÖRNEK SONUÇLAR:")
print("-" * 50)
for i in range(min(5, len(df_test))):
    print(f"Girdi : {df_test.iloc[i]['input_text']}")
    print(f"Çıktı : {df_test.iloc[i]['model_output']}")
    print("-" * 50)

# Dosyaya yaz
df_test.to_csv(OUTPUT_FILE, index=False)
print(f"💾 Tüm sonuçlar '{OUTPUT_FILE}' dosyasına kaydedildi. İndirebilirsin!")

✅ Drive zaten bağlı.
⚙️ Çalışma cihazı: cuda
📂 Model yükleniyor: /content/drive/MyDrive/grammer_model_final_fix2...


The tokenizer you are loading from '/content/drive/MyDrive/grammer_model_final_fix2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ Model başarıyla yüklendi! (Checkpoint-470)
📄 Test dosyası okundu: 1017 satır.
🚀 Düzeltme işlemi başlıyor...


İşleniyor: 100%|██████████| 64/64 [02:33<00:00,  2.40s/it]


👀 ÖRNEK SONUÇLAR:
--------------------------------------------------
Girdi : Guingamp, maça 8. dakikada attığı golle 1-0 önde başlasa da, arka arkaya gelen gollere engel olamadı.
Çıktı : Guingamp, maça 8. dakikada attığı golle 1-0 önde başlasada, arka arkaya gelen gollere engel olamadı.
--------------------------------------------------
Girdi : Guingamp, maça 8. dakikada attığı golle 1-0 önde başlasada, arka arkaya gelen gollere engel olamadı.
Çıktı : Guingamp, maça 8. dakikada attığı golle 1-0 önde başlasada, arka arkaya gelen gollere engel olamadı.
--------------------------------------------------
Girdi : Dünyanın birçok ekonomisi, gelişmiş ülkeler, Avrupa ekonomisi, Avrupa Birliği ekonomide daralırken ya da çok cüzi büyümeler kaydederken biz yüzde 2,2 büyüme oranıyla çok farklı bir yerde duruyoruz.
Çıktı : Dünyanın birçok ekonomisi, gelişmiş ülkeler, Avrupa ekonomisi, Avrupa Birliği ekonomide daralırken ya da çok cüzi büyümeler kaydederken biz yüzde 2,2 büyüme oranıyla çok farklı 

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------------------------------------------------
# 1. AYARLAR
# ---------------------------------------------------------
# Drive'daki checkpoint yolunu buraya yapıştır:
MODEL_PATH = "/content/drive/MyDrive/grammer_model_fix2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Çalışma cihazı: {device}")

# ---------------------------------------------------------
# 2. MODELİ YÜKLE
# ---------------------------------------------------------
print(f"📂 Model yükleniyor: {MODEL_PATH}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device)
    model.eval()
    print("✅ Model başarıyla yüklendi!")
except Exception as e:
    print(f"❌ Model yüklenirken hata oluştu: {e}")
    exit()

# ---------------------------------------------------------
# 3. TEST CÜMLELERİ (BURAYI İSTEDİĞİN GİBİ DOLDUR)
# ---------------------------------------------------------
test_cumleleri = [
    "Ben eve yada  uyuyabileceğim bir yere gitmek istiyorum.",
    "Bugün birazda onunla ilgilenmeliyim.",
    "Karşılıklı sayılarla devam eden maçta periyodun ilk 5 dakikasıda 15-16 Trabzonspor üstünlüğünde geçildi.",
    "Onlarıda yapacağız.",
    "Kimin ne dediğini bende biliyorum.",
    "Hiçbir şey benide yıldıramaz.",
]

# ---------------------------------------------------------
# 4. TOPLU DÜZELTME KODU
# ---------------------------------------------------------
def duzelt(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=5,          # Kalite için beam search
            repetition_penalty=1.2,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*60)
print(f"{'GİRDİ (HATALI)':<30} | {'ÇIKTI (MODEL)':<30}")
print("="*60)

for cumle in test_cumleleri:
    sonuc = duzelt(cumle)
    print(f"{cumle:<30} | {sonuc:<30}")

print("="*60)

⚙️ Çalışma cihazı: cuda
📂 Model yükleniyor: /content/drive/MyDrive/grammer_model_fix2...
❌ Model yüklenirken hata oluştu: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/grammer_model_fix2'. Use `repo_type` argument if needed.

GİRDİ (HATALI)                 | ÇIKTI (MODEL)                 
Ben eve yada  uyuyabileceğim bir yere gitmek istiyorum. | Ben eve ya da uyuyabileceğim bir yere gitmek istiyorum.
Bugün birazda onunla ilgilenmeliyim. | Bugün birazda onunla ilgilenmeliyim.
Karşılıklı sayılarla devam eden maçta periyodun ilk 5 dakikasıda 15-16 Trabzonspor üstünlüğünde geçildi. | Karşılıklı sayılarla devam eden maçta periyodun ilk 5 dakikasında 15-16 Trabzonspor üstünlüğünde geçildi.
Onlarıda yapacağız.            | Onları da yapacağız.          
Kimin ne dediğini bende biliyorum. | Kimin ne dediğini bende biliyorum.
Hiçbir şey benide yıldıramaz.  | Hiçbir şey beni de yıldıramaz.


In [1]:
#BU MODEL DAHA BAŞARILI DURUYOR...

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------------------------------------------------
# 1. AYARLAR
# ---------------------------------------------------------
# Drive'daki checkpoint yolunu buraya yapıştır:
MODEL_PATH = "/content/drive/MyDrive/Egitilmis_Grammar_Modeli/checkpoint-636"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚙️ Çalışma cihazı: {device}")

# ---------------------------------------------------------
# 2. MODELİ YÜKLE
# ---------------------------------------------------------
print(f"📂 Model yükleniyor: {MODEL_PATH}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device)
    model.eval()
    print("✅ Model başarıyla yüklendi!")
except Exception as e:
    print(f"❌ Model yüklenirken hata oluştu: {e}")
    exit()

# ---------------------------------------------------------
# 3. TEST CÜMLELERİ (BURAYI İSTEDİĞİN GİBİ DOLDUR)
# ---------------------------------------------------------
test_cumleleri = [
    "Ben eve yada  uyuyabileceğim bir yere gitmek istiyorum.",
    "Bugün birazda onunla ilgilenmeliyim.",
    "Karşılıklı sayılarla devam eden maçta periyodun ilk 5 dakikasıda 15-16 Trabzonspor üstünlüğünde geçildi.",
    "Onlarıda yapacağız.",
    "Kimin ne dediğini bende biliyorum.",
    "Hiçbir şey benide yıldıramaz.",
]

# ---------------------------------------------------------
# 4. TOPLU DÜZELTME KODU
# ---------------------------------------------------------
def duzelt(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=5,          # Kalite için beam search
            repetition_penalty=1.2,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "="*60)
print(f"{'GİRDİ (HATALI)':<30} | {'ÇIKTI (MODEL)':<30}")
print("="*60)

for cumle in test_cumleleri:
    sonuc = duzelt(cumle)
    print(f"{cumle:<30} | {sonuc:<30}")

print("="*60)

⚙️ Çalışma cihazı: cuda
📂 Model yükleniyor: /content/drive/MyDrive/Egitilmis_Grammar_Modeli/checkpoint-636...
✅ Model başarıyla yüklendi!

GİRDİ (HATALI)                 | ÇIKTI (MODEL)                 
Ben eve yada  uyuyabileceğim bir yere gitmek istiyorum. | Ben eve ya dauyuyabileceğim bir yere gitmek istiyorum.
Bugün birazda onunla ilgilenmeliyim. | Bugün biraz da onunla ilgilenmeliyim.
Karşılıklı sayılarla devam eden maçta periyodun ilk 5 dakikasıda 15-16 Trabzonspor üstünlüğünde geçildi. | Karşılıklı sayılarla devam eden maçta periyodun ilk 5 dakikası da 15-16 Trabzonspor üstünlüğünde geçildi.
Onlarıda yapacağız.            | Onları da yapacağız.          
Kimin ne dediğini bende biliyorum. | Kimin ne dediğini ben de biliyorum.
Hiçbir şey benide yıldıramaz.  | Hiçbir şey beni de yıldıramaz.
